# Calibrated Ensemble Submission
This version includes: 
* TF-IDF baseline
* DeBERTa-v3-small cross-encoder
* 2×T4 DataParallel support
* prior smoothing
* tie calibration
* submission.csv generation

## 01. Setup and Conifg 

In [1]:
import os
import json
import gc
import math
import random
import warnings
from pathlib import Path

In [2]:
import numpy as np
import pandas as pd

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    log_loss,
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

In [5]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

In [6]:
warnings.filterwarnings("ignore")

In [7]:
RANDOM_STATE = 42

In [8]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [9]:
seed_everything(RANDOM_STATE)

In [10]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [11]:
print("Device:", DEVICE)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Device: cuda
Torch version: 2.10.0+cu128
CUDA available: True


In [12]:
if torch.cuda.is_available():
    print("CUDA device count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))
    print("Active GPU:", torch.cuda.get_device_name(0))
    print("CUDA device capability:", torch.cuda.get_device_capability(0))
    print("PyTorch CUDA version:", torch.version.cuda)

CUDA device count: 2
0 Tesla T4
1 Tesla T4
Active GPU: Tesla T4
CUDA device capability: (7, 5)
PyTorch CUDA version: 12.8


In [13]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 500)

In [14]:
# Set to 2000 or 10000 for debug.
# Set to None for real run.
DEBUG_N_ROWS = None
# DEBUG_N_ROWS = 10000

In [15]:
VALID_SIZE = 0.15

In [16]:
MAX_LENGTH = 512

In [17]:
# DataParallel-friendly settings for 2x T4.
# Same effective batch size as:
# BATCH_SIZE = 2, GRAD_ACCUM_STEPS = 4
BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 2

In [18]:
EPOCHS = 2

In [19]:
LEARNING_RATE = 2e-6
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
MAX_GRAD_NORM = 1.0

In [20]:
USE_AMP = False
USE_DATA_PARALLEL = True
USE_SWAP_AUGMENTATION = True

In [21]:
RUN_FINAL_FULL_DATA_DEBERTA_TRAINING = False

In [22]:
TFIDF_MAX_FEATURES = 300_000
TFIDF_C = 0.5

In [23]:
DATA_DIR = Path("/kaggle/input/competitions/llm-classification-finetuning")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"

In [24]:
MODEL_PATH = Path('/kaggle/input/models/mtnash/deberta-v3-small/transformers/default/1')

In [25]:
print({
    "DEBUG_N_ROWS": DEBUG_N_ROWS,
    "VALID_SIZE": VALID_SIZE,
    "MAX_LENGTH": MAX_LENGTH,
    "BATCH_SIZE": BATCH_SIZE,
    "EVAL_BATCH_SIZE": EVAL_BATCH_SIZE,
    "GRAD_ACCUM_STEPS": GRAD_ACCUM_STEPS,
    "EPOCHS": EPOCHS,
    "LEARNING_RATE": LEARNING_RATE,
    "USE_AMP": USE_AMP,
    "USE_DATA_PARALLEL": USE_DATA_PARALLEL,
    "USE_SWAP_AUGMENTATION": USE_SWAP_AUGMENTATION,
    "RUN_FINAL_FULL_DATA_DEBERTA_TRAINING": RUN_FINAL_FULL_DATA_DEBERTA_TRAINING,
})

{'DEBUG_N_ROWS': None, 'VALID_SIZE': 0.15, 'MAX_LENGTH': 512, 'BATCH_SIZE': 4, 'EVAL_BATCH_SIZE': 8, 'GRAD_ACCUM_STEPS': 2, 'EPOCHS': 2, 'LEARNING_RATE': 2e-06, 'USE_AMP': False, 'USE_DATA_PARALLEL': True, 'USE_SWAP_AUGMENTATION': True, 'RUN_FINAL_FULL_DATA_DEBERTA_TRAINING': False}


In [26]:
print("Using input directory:", DATA_DIR)
print("Train path:", TRAIN_PATH)
print("Test path:", TEST_PATH)
print("Sample submission path:", SAMPLE_SUB_PATH)

Using input directory: /kaggle/input/competitions/llm-classification-finetuning
Train path: /kaggle/input/competitions/llm-classification-finetuning/train.csv
Test path: /kaggle/input/competitions/llm-classification-finetuning/test.csv
Sample submission path: /kaggle/input/competitions/llm-classification-finetuning/sample_submission.csv


In [27]:
print("Using model path:", MODEL_PATH)
print("Files in model path:")
print([p.name for p in Path(MODEL_PATH).iterdir()][:50])

Using model path: /kaggle/input/models/mtnash/deberta-v3-small/transformers/default/1
Files in model path:
['spm.model', 'config.json', 'tf_model.h5', 'tokenizer_config.json', 'pytorch_model.bin']


## 02. Load Data 

In [28]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUB_PATH)

In [29]:
if DEBUG_N_ROWS is not None:
    train = train.sample(DEBUG_N_ROWS, random_state=RANDOM_STATE).reset_index(drop=True)

In [30]:
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)

Train shape: (57477, 9)
Test shape: (3, 4)
Sample submission shape: (3, 4)


In [31]:
display(train.head())
display(test.head())
display(sample_submission.head())

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain percentage of females on managerial positions?"",""OK, does pineapple belong on a pizza? Relax and give me fun answer.""]","[""The question of whether it is morally right to aim for a certain percentage of females in managerial positions is a complex ethical issue that involves considerations of fairness, equality, diversity, and discrimination.\n\nHere are some arguments in favor of and against such policies:\n\n**Arguments in favor:**\n\n1. **Correcting Historical Inequities:** Women have historically been underrepresented in leadership roles due to various cultural, institutional, and social barriers. Aiming fo...","[""As an AI, I don't have personal beliefs or opinions. However, I can tell you that the question of gender quotas in managerial positions is a complex one and positions can vary. Advocates argue that such measures can help correct historical imbalances, promote diversity, and may lead to better decision-making. Critics may argue that positions should be based purely on merit, and that quotas could potentially lead to tokenism or unqualified candidates being promoted. Morality can be subjecti...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage license and marriage certificate?"",""How can I get both of them as quick as possible in California "",""What is the minimal time to get them? 1 day or 10 days?""]","[""A marriage license is a legal document that allows a couple to get married. It is issued by a government agency, such as a county clerk's office or a state government, and is valid for a certain period of time, usually one year. After the marriage has taken place, the couple must obtain a marriage certificate, which is a document that records the marriage and is used to prove that the marriage took place. The marriage certificate is usually issued by the same government agency that issued ...","[""A marriage license and a marriage certificate are two different legal documents that have separate purposes.\n\n1. Marriage License: A marriage license is a legal document that gives a couple permission to get married. It's usually obtained from local government or court officials before the wedding ceremony takes place. The couple is required to meet certain criteria, such as being of a certain age or not being closely related. Once the license is issued, there's often a waiting period be...",0,1,0
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call a function?""]","[""Function calling is the process of invoking or executing a function in a programming language. To call a function, you need to use the function's name followed by parentheses \""()\"". Within the parentheses, you can pass arguments if the function requires any.\n\nHere is an example of how to call a function in Python:\n\n```python\n# Define a function\ndef greet(name):\n print(\""Hello, \"" + name + \""!\"")\n\n# Call the function\ngreet(\""John\"")\n```\n\nIn this example, the function \""gree...","[""Function calling is the process of invoking a function in a program. A function is a block of code that performs a specific task and can be reused multiple times in a program. Functions help to organize and modularize code, making it more readable and maintainable.\n\nTo call a function, you simply write the name of the function followed by parentheses () containing any arguments or parameters that the function requires. Here's an example:\n```\nfunction greet(name) {\n console.log(\""Hell...",0,0,1
3,96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare category? I want to build a classifier.""]","[""Creating a test set for a very rare category can be challenging, as it may not be possible to collect a sufficient number of examples for the rare category. However, there 

,id,prompt,response_a,response_b
0,136060,"[""I have three oranges today, I ate an orange yesterday. How many oranges do I have?""]","[""You have two oranges today.""]","[""You still have three oranges. Eating an orange yesterday does not affect the number of oranges you have today.""]"
1,211333,"[""You are a mediator in a heated political debate between two opposing parties. Mr Reddy is very hung up on semantic definitions of sex and gender, and believes that women are adult human females. Meanwhile Ms Blue is extremely fluid with definitions and does not care about truth. He (Ms blue uses he\/him pronouns) insists that anybody can be any gender, gametes don't mean anything, and that men can get pregnant. You, Mr Goddy are tasked with helping them both find a middle ground.""]","[""Thank you for sharing the details of the situation. As a mediator, I understand the importance of finding a middle ground that both parties can agree upon.\n\nIt is important to recognize that language and definitions can be complex and can vary depending on cultural and societal norms. At the same time, it is also important to acknowledge the biological differences between males and females.\n\nOne possible approach to finding a middle ground could be to focus on respecting individuals' s...","[""Mr Reddy and Ms Blue both have valid points in their arguments. On one hand, Mr Reddy is correct that women are adult human females, as defined by their biological and physical characteristics. However, Ms Blue is right that gender is a more fluid concept and that people can identify with any gender they choose. A compromise could be that gender is defined by an individual's self-identification, while sex is defined by their biological characteristics.""]"
2,1233961,"[""How to initialize the classification head when I do transfer learning. For example, I have a pre-trained vision transformer on ImageNet, and now I want to finetune it on StanfordCars"",""I want to do full finetuning""]","[""When you want to initialize the classification head for transfer learning, you can follow these steps:\n\n 1. Load the pre-trained model: First, load the pre-trained vision transformer model on ImageNet. You can use popular deep learning libraries like PyTorch or TensorFlow to load the model.\n 2. Freeze the pre-trained layers: Next, you need to freeze the pre-trained layers of the model so that they don't get updated during the training process. This is because the pre-trained layers ha...","[""To initialize the classification head when performing transfer learning, follow these steps:\n\n1. Load the pre-trained vision transformer model that was trained on ImageNet.\n2. Replace the existing classification head with a new one suitable for the StanfordCars dataset.\n3. Initialize the weights of the new classification head with random values or pre-trained weights if available.\n4. Freeze the weights of the vision transformer backbone to prevent them from being updated during traini..."


,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.333333,0.333333,0.333333
1,211333,0.333333,0.333333,0.333333
2,1233961,0.333333,0.333333,0.333333


## 03. Column Checks

In [32]:
required_train_cols = {
    "id",
    "prompt",
    "response_a",
    "response_b",
    "winner_model_a",
    "winner_model_b",
    "winner_tie",
}

In [33]:
required_test_cols = {
    "id",
    "prompt",
    "response_a",
    "response_b",
}

In [34]:
missing_train = required_train_cols - set(train.columns)
missing_test = required_test_cols - set(test.columns)

In [35]:
if missing_train:
    raise ValueError(f"Missing required train columns: {missing_train}")
if missing_test:
    raise ValueError(f"Missing required test columns: {missing_test}")
print("Column checks passed.")

Column checks passed.


## 04. Parse Text Fields 

In [36]:
def parse_text_field(x):
    """
    Safely parse text fields that may be JSON-encoded lists.

    Examples:
    '["hello", "world"]' -> 'hello\\nworld'
    plain text -> plain text
    NaN -> ''
    """
    if pd.isna(x):
        return ""

    if not isinstance(x, str):
        return str(x)

    try:
        value = json.loads(x)

        if isinstance(value, list):
            return "\n".join("" if item is None else str(item) for item in value)

        if value is None:
            return ""

        return str(value)

    except Exception:
        return x

In [37]:
for col in ["prompt", "response_a", "response_b"]:
    train[f"{col}_parsed"] = train[col].apply(parse_text_field)
    test[f"{col}_parsed"] = test[col].apply(parse_text_field)

In [38]:
display(train[["prompt_parsed", "response_a_parsed", "response_b_parsed"]].head(2))

,prompt_parsed,response_a_parsed,response_b_parsed
0,"Is it morally right to try to have a certain percentage of females on managerial positions?\nOK, does pineapple belong on a pizza? Relax and give me fun answer.","The question of whether it is morally right to aim for a certain percentage of females in managerial positions is a complex ethical issue that involves considerations of fairness, equality, diversity, and discrimination.\n\nHere are some arguments in favor of and against such policies:\n\n**Arguments in favor:**\n\n1. **Correcting Historical Inequities:** Women have historically been underrepresented in leadership roles due to various cultural, institutional, and social barriers. Aiming for ...","As an AI, I don't have personal beliefs or opinions. However, I can tell you that the question of gender quotas in managerial positions is a complex one and positions can vary. Advocates argue that such measures can help correct historical imbalances, promote diversity, and may lead to better decision-making. Critics may argue that positions should be based purely on merit, and that quotas could potentially lead to tokenism or unqualified candidates being promoted. Morality can be subjective..."
1,What is the difference between marriage license and marriage certificate?\nHow can I get both of them as quick as possible in California \nWhat is the minimal time to get them? 1 day or 10 days?,"A marriage license is a legal document that allows a couple to get married. It is issued by a government agency, such as a county clerk's office or a state government, and is valid for a certain period of time, usually one year. After the marriage has taken place, the couple must obtain a marriage certificate, which is a document that records the marriage and is used to prove that the marriage took place. The marriage certificate is usually issued by the same government agency that issued th...","A marriage license and a marriage certificate are two different legal documents that have separate purposes.\n\n1. Marriage License: A marriage license is a legal document that gives a couple permission to get married. It's usually obtained from local government or court officials before the wedding ceremony takes place. The couple is required to meet certain criteria, such as being of a certain age or not being closely related. Once the license is issued, there's often a waiting period befo..."


## 05. Create Target Labels 

In [39]:
target_cols = ["winner_model_a", "winner_model_b", "winner_tie"]

In [40]:
train["target"] = train[target_cols].values.argmax(axis=1)

In [41]:
label_names = {
    0: "winner_model_a",
    1: "winner_model_b",
    2: "winner_tie",
}

In [42]:
print("Target distribution:")
target_dist = train["target"].value_counts(normalize=True).sort_index()

Target distribution:


In [43]:
for idx, value in target_dist.items():
    print(f"{idx} ({label_names[idx]}): {value:.4f}")

0 (winner_model_a): 0.3491
1 (winner_model_b): 0.3419
2 (winner_tie): 0.3090


In [44]:
display(train[target_cols + ["target"]].head())

,winner_model_a,winner_model_b,winner_tie,target
0,1,0,0,0
1,0,1,0,1
2,0,0,1,2
3,1,0,0,0
4,0,1,0,1


## 06. Build Cross-Encoder Text 

In [45]:
def build_comparison_text(df):
    return (
        "Prompt:\n" + df["prompt_parsed"].fillna("") +
        "\n\nResponse A:\n" + df["response_a_parsed"].fillna("") +
        "\n\nResponse B:\n" + df["response_b_parsed"].fillna("")
    )

In [46]:
train["text"] = build_comparison_text(train)
test["text"] = build_comparison_text(test)

In [47]:
print(train["text"].iloc[0][:2000])

Prompt:
Is it morally right to try to have a certain percentage of females on managerial positions?
OK, does pineapple belong on a pizza? Relax and give me fun answer.

Response A:
The question of whether it is morally right to aim for a certain percentage of females in managerial positions is a complex ethical issue that involves considerations of fairness, equality, diversity, and discrimination.

Here are some arguments in favor of and against such policies:

**Arguments in favor:**

1. **Correcting Historical Inequities:** Women have historically been underrepresented in leadership roles due to various cultural, institutional, and social barriers. Aiming for a specific percentage can be seen as a corrective measure to address past and ongoing discrimination.

2. **Promoting Diversity:** Diverse leadership teams can enhance decision-making and represent a broader range of perspectives. This can lead to better outcomes for organizations and society as a whole.

3. **Equality of Oppor

## 07. A/B Swap Augmentation

In [48]:
def create_swapped_training_data(df):
    """
    Create augmented rows where response_a and response_b are swapped.

    Label mapping:
    0 -> 1
    1 -> 0
    2 -> 2
    """
    swapped = df.copy()

    swapped["response_a_parsed"] = df["response_b_parsed"].values
    swapped["response_b_parsed"] = df["response_a_parsed"].values

    swapped["text"] = build_comparison_text(swapped)

    swapped["target"] = df["target"].map({
        0: 1,
        1: 0,
        2: 2,
    }).astype(int).values

    return swapped

In [49]:
swapped_preview = create_swapped_training_data(train.head(3))

In [50]:
print("Original targets:", train.loc[:2, "target"].tolist())
print("Swapped targets:", swapped_preview["target"].tolist())

Original targets: [0, 1, 2]
Swapped targets: [1, 0, 2]


## 08. Shared Train/Validation Split 

In [51]:
train_base, valid_base = train_test_split(
    train,
    test_size=VALID_SIZE,
    random_state=RANDOM_STATE,
    stratify=train["target"],
)

In [52]:
if USE_SWAP_AUGMENTATION:
    train_base_swapped = create_swapped_training_data(train_base)

    train_model_df = pd.concat(
        [
            train_base[["text", "target"]],
            train_base_swapped[["text", "target"]],
        ],
        axis=0,
        ignore_index=True,
    )
else:
    train_model_df = train_base[["text", "target"]].copy()

In [53]:
valid_model_df = valid_base[["text", "target"]].copy()

In [54]:
y_valid = valid_model_df["target"].values

In [55]:
print("Train rows before augmentation:", train_base.shape[0])
print("Train rows after augmentation:", train_model_df.shape[0])
print("Validation rows:", valid_model_df.shape[0])

Train rows before augmentation: 48855
Train rows after augmentation: 97710
Validation rows: 8622


In [56]:
print("\nTrain target distribution:")
print(train_model_df["target"].value_counts(normalize=True).sort_index())


Train target distribution:
target
0    0.345492
1    0.345492
2    0.309016
Name: proportion, dtype: float64


In [57]:
print("\nValidation target distribution:")
print(valid_model_df["target"].value_counts(normalize=True).sort_index())


Validation target distribution:
target
0    0.349107
1    0.341916
2    0.308977
Name: proportion, dtype: float64


## 09. TF-IDF + Logistic Regression Baseline

In [58]:
def clip_and_normalize(preds, eps=1e-5):
    preds = np.clip(preds, eps, 1 - eps)
    return preds / preds.sum(axis=1, keepdims=True)

In [59]:
def make_tfidf_logreg_model(C=0.5, max_features=300_000):
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            analyzer="word",
            ngram_range=(1, 2),
            max_features=max_features,
            min_df=2,
            max_df=0.95,
            sublinear_tf=True,
            dtype=np.float32,
        )),
        ("clf", LogisticRegression(
            C=C,
            solver="saga",
            multi_class="multinomial",
            max_iter=1000,
            n_jobs=-1,
            random_state=RANDOM_STATE,
            verbose=0,
        )),
    ])

In [60]:
tfidf_model = make_tfidf_logreg_model(
    C=TFIDF_C,
    max_features=TFIDF_MAX_FEATURES,
)

In [61]:
tfidf_model.fit(
    train_model_df["text"].values,
    train_model_df["target"].values,
)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(dtype=<class 'numpy.float32'>, max_df=0.95,
                                 max_features=300000, min_df=2,
                                 ngram_range=(1, 2), strip_accents='unicode',
                                 sublinear_tf=True)),
                ('clf',
                 LogisticRegression(C=0.5, max_iter=1000,
                                    multi_class='multinomial', n_jobs=-1,
                                    random_state=42, solver='saga'))])

In [62]:
tfidf_valid_probs = tfidf_model.predict_proba(valid_model_df["text"].values)
tfidf_valid_probs = clip_and_normalize(tfidf_valid_probs)

In [63]:
tfidf_valid_logloss = log_loss(y_valid, tfidf_valid_probs, labels=[0, 1, 2])
tfidf_valid_acc = accuracy_score(y_valid, tfidf_valid_probs.argmax(axis=1))

In [64]:
print(f"TF-IDF validation log loss: {tfidf_valid_logloss:.6f}")
print(f"TF-IDF validation accuracy: {tfidf_valid_acc:.6f}")

TF-IDF validation log loss: 1.082605
TF-IDF validation accuracy: 0.388541


## 10. Toeknizer and Dataset 

In [65]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    use_fast=False,
)

The tokenizer you are loading from '/kaggle/input/models/mtnash/deberta-v3-small/transformers/default/1' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [66]:
print("Tokenizer loaded.")
print("Tokenizer model max length:", tokenizer.model_max_length)

Tokenizer loaded.
Tokenizer model max length: 1000000000000000019884624838656


In [67]:
class PreferenceDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_length=512):
        self.texts = list(texts)
        self.labels = None if labels is None else list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        encoded = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None,
        )

        item = {
            "input_ids": torch.tensor(encoded["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(encoded["attention_mask"], dtype=torch.long),
        }

        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item

In [68]:
def collate_fn(batch):
    input_ids = [x["input_ids"] for x in batch]
    attention_mask = [x["attention_mask"] for x in batch]

    padded = tokenizer.pad(
        {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
        },
        padding=True,
        return_tensors="pt",
    )

    if "labels" in batch[0]:
        padded["labels"] = torch.stack([x["labels"] for x in batch])

    return padded

## 11. Dataloaders 

In [69]:
train_dataset = PreferenceDataset(
    texts=train_model_df["text"].values,
    labels=train_model_df["target"].values,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

In [70]:
valid_dataset = PreferenceDataset(
    texts=valid_model_df["text"].values,
    labels=valid_model_df["target"].values,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

In [71]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True,
)

In [72]:
valid_loader = DataLoader(
    valid_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True,
)

In [73]:
print("Train batches:", len(train_loader))
print("Validation batches:", len(valid_loader))

Train batches: 24428
Validation batches: 1078


## 12. DeBERTa Model, Optimizer, Scheduler

In [74]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    num_labels=3,
    torch_dtype=torch.float32,
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: /kaggle/input/models/mtnash/deberta-v3-small/transformers/default/1
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                      

In [75]:
model.to(DEVICE)

DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-5): 6 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNo

In [76]:
if USE_DATA_PARALLEL and torch.cuda.is_available() and torch.cuda.device_count() > 1:
    print("Using DataParallel on", torch.cuda.device_count(), "GPUs")
    model = torch.nn.DataParallel(model)
else:
    print("Using single device:", DEVICE)

Using DataParallel on 2 GPUs


In [77]:
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

In [78]:
num_update_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
num_training_steps = EPOCHS * num_update_steps_per_epoch
num_warmup_steps = int(WARMUP_RATIO * num_training_steps)

In [79]:
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

In [80]:
scaler = torch.cuda.amp.GradScaler(enabled=False)

In [81]:
base_model = model.module if hasattr(model, "module") else model

In [82]:
print("Training steps:", num_training_steps)
print("Warmup steps:", num_warmup_steps)
print("Model dtype:", next(base_model.parameters()).dtype)

Training steps: 24428
Warmup steps: 1465
Model dtype: torch.float32


## 13. Training Helpers 

In [83]:
def move_batch_to_device(batch, device):
    return {k: v.to(device) for k, v in batch.items()}

In [84]:
def unwrap_model(model):
    return model.module if hasattr(model, "module") else model

In [85]:
def train_one_epoch(model, loader, optimizer, scheduler, scaler=None):
    model.train()

    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    use_amp_now = USE_AMP and DEVICE.type == "cuda"

    for step, batch in enumerate(loader):
        batch = move_batch_to_device(batch, DEVICE)

        if use_amp_now:
            with torch.cuda.amp.autocast():
                outputs = model(**batch)
                loss_raw = outputs.loss

                if loss_raw.ndim > 0:
                    loss_raw = loss_raw.mean()

                if torch.isnan(loss_raw) or torch.isinf(loss_raw):
                    print(f"Bad loss at step {step + 1}: {loss_raw}")
                    print("Labels:", batch["labels"][:10])
                    print("Input IDs min/max:", batch["input_ids"].min().item(), batch["input_ids"].max().item())
                    raise ValueError("NaN or Inf loss detected.")

                loss = loss_raw / GRAD_ACCUM_STEPS

            scaler.scale(loss).backward()

            if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)

                scaler.step(optimizer)
                scaler.update()

                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

        else:
            outputs = model(**batch)
            loss_raw = outputs.loss

            if loss_raw.ndim > 0:
                loss_raw = loss_raw.mean()

            if torch.isnan(loss_raw) or torch.isinf(loss_raw):
                print(f"Bad loss at step {step + 1}: {loss_raw}")
                print("Labels:", batch["labels"][:10])
                print("Input IDs min/max:", batch["input_ids"].min().item(), batch["input_ids"].max().item())
                raise ValueError("NaN or Inf loss detected.")

            loss = loss_raw / GRAD_ACCUM_STEPS
            loss.backward()

            if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)

                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

        total_loss += loss_raw.item()

        if (step + 1) % 250 == 0:
            print(f"Step {step + 1}/{len(loader)} | loss: {total_loss / (step + 1):.5f}")

    return total_loss / len(loader)

In [86]:
@torch.no_grad()
def predict_proba(model, loader):
    model.eval()

    all_probs = []
    all_labels = []

    for batch in loader:
        labels = batch.get("labels")

        batch = move_batch_to_device(batch, DEVICE)

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )

        logits = outputs.logits.detach().float().cpu().numpy()

        exp_logits = np.exp(logits - logits.max(axis=1, keepdims=True))
        probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)

        all_probs.append(probs)

        if labels is not None:
            all_labels.append(labels.cpu().numpy())

    all_probs = np.vstack(all_probs)

    if all_labels:
        all_labels = np.concatenate(all_labels)
        return all_probs, all_labels

    return all_probs, None

## 14. Sanity Check Initial Batch 

In [87]:
batch = next(iter(train_loader))
batch = move_batch_to_device(batch, DEVICE)

In [88]:
with torch.no_grad():
    outputs = model(**batch)
    initial_loss = outputs.loss

    if initial_loss.ndim > 0:
        initial_loss = initial_loss.mean()

In [89]:
print("Initial loss:", initial_loss)
print("Any NaN in logits:", torch.isnan(outputs.logits).any().item())
print("Logits:", outputs.logits[:5])
print("Labels:", batch["labels"][:10])
print("Input IDs min/max:", batch["input_ids"].min().item(), batch["input_ids"].max().item())

Initial loss: tensor(1.1871, device='cuda:0')
Any NaN in logits: False
Logits: tensor([[-0.2506, -0.3674, -0.1154],
        [-0.0793, -0.4960, -0.1480],
        [-0.1665, -0.4611, -0.0159],
        [-0.2778, -0.3863, -0.2447]], device='cuda:0')
Labels: tensor([2, 1, 1, 2], device='cuda:0')
Input IDs min/max: 0 120838


## 15. rain DeBERTa Validation Model

In [90]:
best_valid_logloss = float("inf")
best_state_dict = None

In [91]:
for epoch in range(EPOCHS):
    print("\n" + "=" * 80)
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print("=" * 80)

    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
    )

    valid_probs, valid_labels = predict_proba(model, valid_loader)
    valid_probs = clip_and_normalize(valid_probs)

    valid_logloss = log_loss(valid_labels, valid_probs, labels=[0, 1, 2])
    valid_preds = valid_probs.argmax(axis=1)
    valid_acc = accuracy_score(valid_labels, valid_preds)

    print(f"Train loss: {train_loss:.6f}")
    print(f"DeBERTa validation log loss: {valid_logloss:.6f}")
    print(f"DeBERTa validation accuracy: {valid_acc:.6f}")

    if valid_logloss < best_valid_logloss:
        best_valid_logloss = valid_logloss

        model_to_save = unwrap_model(model)
        best_state_dict = {
            k: v.detach().cpu().clone()
            for k, v in model_to_save.state_dict().items()
        }

        print("New best DeBERTa model saved in memory.")


Epoch 1/2
Step 250/24428 | loss: 1.11947
Step 500/24428 | loss: 1.11803
Step 750/24428 | loss: 1.11173
Step 1000/24428 | loss: 1.10944
Step 1250/24428 | loss: 1.10787
Step 1500/24428 | loss: 1.10659
Step 1750/24428 | loss: 1.10533
Step 2000/24428 | loss: 1.10479
Step 2250/24428 | loss: 1.10410
Step 2500/24428 | loss: 1.10368
Step 2750/24428 | loss: 1.10299
Step 3000/24428 | loss: 1.10278
Step 3250/24428 | loss: 1.10217
Step 3500/24428 | loss: 1.10163
Step 3750/24428 | loss: 1.10137
Step 4000/24428 | loss: 1.10090
Step 4250/24428 | loss: 1.10022
Step 4500/24428 | loss: 1.09959
Step 4750/24428 | loss: 1.09935
Step 5000/24428 | loss: 1.09888
Step 5250/24428 | loss: 1.09860
Step 5500/24428 | loss: 1.09803
Step 5750/24428 | loss: 1.09780
Step 6000/24428 | loss: 1.09734
Step 6250/24428 | loss: 1.09667
Step 6500/24428 | loss: 1.09609
Step 6750/24428 | loss: 1.09596
Step 7000/24428 | loss: 1.09600
Step 7250/24428 | loss: 1.09521
Step 7500/24428 | loss: 1.09485
Step 7750/24428 | loss: 1.09451


In [92]:
print("\nBest DeBERTa validation log loss:", best_valid_logloss)


Best DeBERTa validation log loss: 1.0475016522131009


In [93]:
if best_state_dict is not None:
    model_to_load = unwrap_model(model)
    model_to_load.load_state_dict(best_state_dict)
    model.to(DEVICE)

In [94]:
deberta_valid_probs, valid_labels = predict_proba(model, valid_loader)
deberta_valid_probs = clip_and_normalize(deberta_valid_probs)

In [95]:
deberta_valid_logloss = log_loss(valid_labels, deberta_valid_probs, labels=[0, 1, 2])
deberta_valid_acc = accuracy_score(valid_labels, deberta_valid_probs.argmax(axis=1))

In [96]:
print(f"Final selected DeBERTa validation log loss: {deberta_valid_logloss:.6f}")
print(f"Final selected DeBERTa validation accuracy: {deberta_valid_acc:.6f}")

Final selected DeBERTa validation log loss: 1.047502
Final selected DeBERTa validation accuracy: 0.451287


## 16. Tune Ensemble Weights

In [97]:
class_priors = train_base[["winner_model_a", "winner_model_b", "winner_tie"]].mean().values
prior_valid_probs = np.tile(class_priors, (len(valid_model_df), 1))

In [98]:
print("Class priors:", class_priors)

Class priors: [0.34907379 0.34190973 0.30901648]


In [99]:
results = []

In [100]:
candidate_weights = []

In [101]:
for w_deberta in np.arange(0.60, 1.01, 0.05):
    for w_tfidf in np.arange(0.00, 0.31, 0.05):
        w_prior = 1.0 - w_deberta - w_tfidf

        if w_prior < -1e-9:
            continue

        w_prior = max(0.0, w_prior)

        candidate_weights.append((
            round(w_deberta, 2),
            round(w_tfidf, 2),
            round(w_prior, 2),
        ))

In [102]:
candidate_weights += [
    (1.00, 0.00, 0.00),
    (0.95, 0.00, 0.05),
    (0.95, 0.05, 0.00),
    (0.90, 0.05, 0.05),
    (0.90, 0.10, 0.00),
    (0.85, 0.10, 0.05),
    (0.85, 0.15, 0.00),
    (0.80, 0.15, 0.05),
    (0.75, 0.20, 0.05),
]

In [103]:
candidate_weights = sorted(set(candidate_weights), reverse=True)

In [104]:
for w_deberta, w_tfidf, w_prior in candidate_weights:
    total = w_deberta + w_tfidf + w_prior

    if abs(total - 1.0) > 1e-6:
        continue

    ensemble_probs = (
        w_deberta * deberta_valid_probs +
        w_tfidf * tfidf_valid_probs +
        w_prior * prior_valid_probs
    )

    ensemble_probs = clip_and_normalize(ensemble_probs)

    score = log_loss(valid_labels, ensemble_probs, labels=[0, 1, 2])
    acc = accuracy_score(valid_labels, ensemble_probs.argmax(axis=1))

    results.append({
        "w_deberta": w_deberta,
        "w_tfidf": w_tfidf,
        "w_prior": w_prior,
        "log_loss": score,
        "accuracy": acc,
    })

In [105]:
results_df = pd.DataFrame(results).sort_values("log_loss").reset_index(drop=True)

In [106]:
display(results_df.head(20))

,w_deberta,w_tfidf,w_prior,log_loss,accuracy
0,0.80,0.20,0.00,1.044715,0.454303
1,0.75,0.25,0.00,1.044825,0.455579
2,0.85,0.15,0.00,1.044916,0.454187
3,0.80,0.15,0.05,1.044994,0.454651
4,0.75,0.20,0.05,1.045026,0.453955
5,0.70,0.30,0.00,1.045238,0.455463
6,0.85,0.10,0.05,1.045275,0.452331
7,0.70,0.25,0.05,1.045362,0.454999
8,0.75,0.15,0.10,1.045400,0.454419
9,0.90,0.10,0.00,1.045437,0.452795


In [107]:
best = results_df.iloc[0]

In [108]:
BEST_W_DEBERTA = float(best["w_deberta"])
BEST_W_TFIDF = float(best["w_tfidf"])
BEST_W_PRIOR = float(best["w_prior"])
BEST_ENSEMBLE_LOGLOSS = float(best["log_loss"])

In [109]:
print("Best weights:")
print({
    "BEST_W_DEBERTA": BEST_W_DEBERTA,
    "BEST_W_TFIDF": BEST_W_TFIDF,
    "BEST_W_PRIOR": BEST_W_PRIOR,
    "BEST_ENSEMBLE_LOGLOSS": BEST_ENSEMBLE_LOGLOSS,
})

Best weights:
{'BEST_W_DEBERTA': 0.8, 'BEST_W_TFIDF': 0.2, 'BEST_W_PRIOR': 0.0, 'BEST_ENSEMBLE_LOGLOSS': 1.0447153726289389}


## 17. Optional Tie Calibration

In [110]:
base_ensemble_valid_probs = (
    BEST_W_DEBERTA * deberta_valid_probs +
    BEST_W_TFIDF * tfidf_valid_probs +
    BEST_W_PRIOR * prior_valid_probs
)

In [111]:
base_ensemble_valid_probs = clip_and_normalize(base_ensemble_valid_probs)

In [112]:
tie_results = []

In [113]:
for tie_multiplier in [
    0.85,
    0.90,
    0.95,
    1.00,
    1.03,
    1.05,
    1.07,
    1.10,
    1.12,
    1.15,
    1.20,
    1.25,
]:
    adjusted = base_ensemble_valid_probs.copy()
    adjusted[:, 2] *= tie_multiplier
    adjusted = clip_and_normalize(adjusted)

    score = log_loss(valid_labels, adjusted, labels=[0, 1, 2])
    acc = accuracy_score(valid_labels, adjusted.argmax(axis=1))

    tie_results.append({
        "tie_multiplier": tie_multiplier,
        "log_loss": score,
        "accuracy": acc,
        "mean_tie_prob": adjusted[:, 2].mean(),
    })

In [114]:
tie_results_df = pd.DataFrame(tie_results).sort_values("log_loss").reset_index(drop=True)

In [115]:
display(tie_results_df)

,tie_multiplier,log_loss,accuracy,mean_tie_prob
0,1.00,1.044715,0.454303,0.311381
1,0.95,1.044856,0.456275,0.301137
2,1.03,1.044875,0.455115,0.317368
3,1.05,1.045074,0.456159,0.321296
4,1.07,1.045343,0.455347,0.325174
5,0.90,1.045567,0.457434,0.290546
6,1.10,1.045870,0.455231,0.330900
7,1.12,1.046299,0.452911,0.334658
8,0.85,1.046935,0.457203,0.279586
9,1.15,1.047051,0.451403,0.340208


In [116]:
best_tie = tie_results_df.iloc[0]

In [117]:
BEST_TIE_MULTIPLIER = float(best_tie["tie_multiplier"])
BEST_TIE_LOGLOSS = float(best_tie["log_loss"])

In [118]:
print("Best tie multiplier:", BEST_TIE_MULTIPLIER)
print("Best tie-calibrated validation log loss:", BEST_TIE_LOGLOSS)

Best tie multiplier: 1.0
Best tie-calibrated validation log loss: 1.0447153724409026


## 18. Final Validation Diagnostics

In [119]:
final_valid_probs = base_ensemble_valid_probs.copy()
final_valid_probs[:, 2] *= BEST_TIE_MULTIPLIER
final_valid_probs = clip_and_normalize(final_valid_probs)

In [120]:
final_valid_preds = final_valid_probs.argmax(axis=1)

In [121]:
final_logloss = log_loss(valid_labels, final_valid_probs, labels=[0, 1, 2])
final_acc = accuracy_score(valid_labels, final_valid_preds)

In [122]:
print(f"TF-IDF valid log loss:   {tfidf_valid_logloss:.6f}")
print(f"DeBERTa valid log loss:  {deberta_valid_logloss:.6f}")
print(f"Ensemble valid log loss: {final_logloss:.6f}")
print(f"Ensemble valid accuracy: {final_acc:.6f}")

TF-IDF valid log loss:   1.082605
DeBERTa valid log loss:  1.047502
Ensemble valid log loss: 1.044715
Ensemble valid accuracy: 0.454303


In [123]:
print("\nClassification Report:")
print(classification_report(
    valid_labels,
    final_valid_preds,
    target_names=["winner_model_a", "winner_model_b", "winner_tie"],
    digits=4,
))


Classification Report:
                precision    recall  f1-score   support

winner_model_a     0.4586    0.5066    0.4815      3010
winner_model_b     0.4505    0.4725    0.4613      2948
    winner_tie     0.4531    0.3750    0.4104      2664

      accuracy                         0.4543      8622
     macro avg     0.4541    0.4514    0.4510      8622
  weighted avg     0.4541    0.4543    0.4526      8622



In [124]:
print("\nConfusion Matrix:")
print(confusion_matrix(valid_labels, final_valid_preds))


Confusion Matrix:
[[1525  884  601]
 [ 950 1393  605]
 [ 850  815  999]]


In [125]:
prob_df = pd.DataFrame(
    final_valid_probs,
    columns=["winner_model_a", "winner_model_b", "winner_tie"],
)

In [126]:
print("\nAverage ensemble predicted probabilities:")
display(prob_df.mean())


Average ensemble predicted probabilities:


winner_model_a    0.337323
winner_model_b    0.351296
winner_tie        0.311381
dtype: float64

In [127]:
print("\nActual validation class distribution:")
display(pd.Series(valid_labels).value_counts(normalize=True).sort_index().rename(index=label_names))


Actual validation class distribution:


winner_model_a    0.349107
winner_model_b    0.341916
winner_tie        0.308977
Name: proportion, dtype: float64

## 19. Test DataLoaders and Predictions 

In [128]:
# TF-IDF test probabilities
tfidf_test_probs = tfidf_model.predict_proba(test["text"].values)
tfidf_test_probs = clip_and_normalize(tfidf_test_probs)

In [129]:
# DeBERTa test probabilities
test_dataset = PreferenceDataset(
    texts=test["text"].values,
    labels=None,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

In [130]:
test_loader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True,
)

In [131]:
deberta_test_probs, _ = predict_proba(model, test_loader)
deberta_test_probs = clip_and_normalize(deberta_test_probs)

In [132]:
print("TF-IDF test probs shape:", tfidf_test_probs.shape)
print("DeBERTa test probs shape:", deberta_test_probs.shape)

TF-IDF test probs shape: (3, 3)
DeBERTa test probs shape: (3, 3)


In [133]:
print("\nFirst DeBERTa test probabilities:")
print(deberta_test_probs[:5])


First DeBERTa test probabilities:
[[0.18504797 0.1399847  0.6749674 ]
 [0.3443831  0.38996062 0.26565626]
 [0.3146943  0.22455958 0.46074614]]


In [134]:
print("\nFirst TF-IDF test probabilities:")
print(tfidf_test_probs[:5])


First TF-IDF test probabilities:
[[0.251798   0.24873422 0.49946776]
 [0.35736546 0.35446548 0.28816906]
 [0.43288115 0.4292947  0.13782416]]


## 20. Final Test Ensemble

In [135]:
full_class_priors = train[["winner_model_a", "winner_model_b", "winner_tie"]].mean().values
prior_test_probs = np.tile(full_class_priors, (len(test), 1))

In [136]:
test_probs = (
    BEST_W_DEBERTA * deberta_test_probs +
    BEST_W_TFIDF * tfidf_test_probs +
    BEST_W_PRIOR * prior_test_probs
)

In [137]:
test_probs[:, 2] *= BEST_TIE_MULTIPLIER
test_probs = clip_and_normalize(test_probs)

In [138]:
print("Full training class priors:")
print(full_class_priors)

Full training class priors:
[0.34907876 0.34191068 0.30901056]


In [139]:
print("\nFinal test probabilities:")
print(test_probs[:5])


Final test probabilities:
[[0.19839796 0.1617346  0.63986744]
 [0.34697956 0.38286161 0.27015883]
 [0.33833166 0.26550662 0.39616172]]


In [140]:
print("\nRow sums:")
print(test_probs.sum(axis=1)[:5])


Row sums:
[1. 1. 1.]


## 21. Create Submission 

In [141]:
submission = pd.DataFrame({
    "id": test["id"],
    "winner_model_a": test_probs[:, 0],
    "winner_model_b": test_probs[:, 1],
    "winner_tie": test_probs[:, 2],
})

In [142]:
submission = submission[[
    "id",
    "winner_model_a",
    "winner_model_b",
    "winner_tie",
]]

In [143]:
display(submission.head())

,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.198398,0.161735,0.639867
1,211333,0.346980,0.382862,0.270159
2,1233961,0.338332,0.265507,0.396162


In [144]:
print("Submission shape:", submission.shape)

Submission shape: (3, 4)


In [145]:
print("\nProbability row sums:")
display(submission[["winner_model_a", "winner_model_b", "winner_tie"]].sum(axis=1).describe())


Probability row sums:


count    3.000000e+00
mean     1.000000e+00
std      7.850462e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64

In [146]:
submission.to_csv("submission.csv", index=False)

In [147]:
print("Saved submission.csv")
display(pd.read_csv("submission.csv").head())

Saved submission.csv


,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.198398,0.161735,0.639867
1,211333,0.346980,0.382862,0.270159
2,1233961,0.338332,0.265507,0.396162


## 22. Summary 

In [148]:
print("=" * 80)
print("Experiment Summary")
print("=" * 80)

print("Settings:")
print({
    "DEBUG_N_ROWS": DEBUG_N_ROWS,
    "MAX_LENGTH": MAX_LENGTH,
    "BATCH_SIZE": BATCH_SIZE,
    "EVAL_BATCH_SIZE": EVAL_BATCH_SIZE,
    "GRAD_ACCUM_STEPS": GRAD_ACCUM_STEPS,
    "EPOCHS": EPOCHS,
    "LEARNING_RATE": LEARNING_RATE,
    "USE_AMP": USE_AMP,
    "USE_DATA_PARALLEL": USE_DATA_PARALLEL,
    "USE_SWAP_AUGMENTATION": USE_SWAP_AUGMENTATION,
})

print("\nValidation scores:")
print(f"TF-IDF log loss:         {tfidf_valid_logloss:.6f}")
print(f"DeBERTa log loss:        {deberta_valid_logloss:.6f}")
print(f"Ensemble log loss:       {final_logloss:.6f}")
print(f"Best tie-cal log loss:   {BEST_TIE_LOGLOSS:.6f}")

print("\nBest ensemble parameters:")
print({
    "BEST_W_DEBERTA": BEST_W_DEBERTA,
    "BEST_W_TFIDF": BEST_W_TFIDF,
    "BEST_W_PRIOR": BEST_W_PRIOR,
    "BEST_TIE_MULTIPLIER": BEST_TIE_MULTIPLIER,
})

print("\nSubmission file created: submission.csv")

Experiment Summary
Settings:
{'DEBUG_N_ROWS': None, 'MAX_LENGTH': 512, 'BATCH_SIZE': 4, 'EVAL_BATCH_SIZE': 8, 'GRAD_ACCUM_STEPS': 2, 'EPOCHS': 2, 'LEARNING_RATE': 2e-06, 'USE_AMP': False, 'USE_DATA_PARALLEL': True, 'USE_SWAP_AUGMENTATION': True}

Validation scores:
TF-IDF log loss:         1.082605
DeBERTa log loss:        1.047502
Ensemble log loss:       1.044715
Best tie-cal log loss:   1.044715

Best ensemble parameters:
{'BEST_W_DEBERTA': 0.8, 'BEST_W_TFIDF': 0.2, 'BEST_W_PRIOR': 0.0, 'BEST_TIE_MULTIPLIER': 1.0}

Submission file created: submission.csv
